# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a FAIR^2 dataset described by a Croissant schema using the `mlcroissant` library. 

### Dataset Source
The dataset's schema is provided at the following Croissant URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

*This dataset captures ordered logistic regression outputs and survey data on socio-demographics, gender roles, knowledge adoption, and rangeland management practices from pastoralist households in Northern Kenya.*


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Explore available record sets and fields. All entities are referenced by their Croissant `@id` values for unambiguous reference.

Let's print a summary of the available record sets, each with its own `@id` and associated fields/columns.

In [ ]:
# List all record sets in the dataset by @id
record_sets = list(dataset.record_sets)
print("Total number of record sets:", len(record_sets))

for record_set in record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    if 'name' in record_set:
        print(f"  Name: {record_set['name']}")
    if 'field' in record_set:
        print("  Fields/Columns @id:")
        for field in record_set['field']:
            print(f"    - {field['@id']}")

### Review a sample record for each record set
Let’s print the first 1–2 records from each available record set using their `@id`.

In [ ]:
# Show a couple of example records from each available record set
for record_set in record_sets:
    rec_id = record_set['@id']
    print(f"\nSample records from Record Set: {rec_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rec_id)):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"  Could not load records for {rec_id}: {e}")

## 3. Data Extraction

Extract records from each record set into pandas DataFrames for analysis. Use the exact `@id` values gathered above.

> **Note:** Replace or add desired `record_set_ids` if more are present or relevant.

In [ ]:
# List of record set @ids available in this dataset
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}
for recset_id in record_set_ids:
    try:
        # Load all rows for this record set
        df = pd.DataFrame(list(dataset.records(record_set=recset_id)))
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records for record set '@id': {recset_id}")
    except Exception as e:
        print(f"Could not load records for record set {recset_id}. Reason: {e}")

# Display info for first (or only) record set
if dataframes:
    main_rs = list(dataframes.keys())[0]
    print(f"\nColumns in record set '{main_rs}':")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)

We perform basic exploratory analysis: filter and normalize a numeric field, and (attempt to) group by a categorical field.

> **All columns must be referenced by their Croissant `@id`.** Please update the field IDs below as shown in section 2 if needed.

*If you don't know the field @ids, refer back to the data overview above.*

In [ ]:
# Select a record set and field(s) for EDA by `@id`
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# Identify a numeric and a groupable (categorical) field by @id
print("Fields available in selected record set:")
for col in df.columns:
    print(f"  - {col}")

# Example: Let's pick a numeric field and a categorical field by @id
# (set these to the actual field/column @ids from your dataset)
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    if group_field_id is None and df[col].dtype == object:
        group_field_id = col

if numeric_field_id is None:
    print("No numeric field found in DataFrame.")
else:
    print(f"Numeric field selected by @id: {numeric_field_id}")
if group_field_id is None:
    print("No groupable field found.")
else:
    print(f"Categorical field selected by @id: {group_field_id}")

# Filtering: keep records where numeric_field > threshold (if any records)
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: (showing top 5)")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional grouping by a categorical/groupable field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id} (top groups):")
        print(grouped_df.head())
else:
    print("Numeric field not found; skipping EDA step.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and, if appropriate, a boxplot by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion

- We loaded a FAIR^2 dataset using a Croissant schema and the `mlcroissant` library.
- All dataset entities (record sets and fields/columns) were referenced by their stable `@id` values.
- We previewed the record sets and columns, loaded them as DataFrames, and performed basic exploration and visualization over a selected record set field.

Further analysis (e.g., deeper statistical review, regression modeling, or cross-record set joins) should always use field and record set `@id` values to ensure correctness and reproducibility.